In [ ]:
import sys
sys.path.insert(0, 'src')

# Gross Code — LER vs Physical Error Rate

This notebook sweeps physical error rate $p$ for the [[144,12,12]] gross code and compares:

1. **Memory experiment** — bare syndrome rounds using `BBCodeSimulator(BB_144_12_12)` + RelayBP
2. **LPU X̄₁ circuit** — full gauging-measurement circuit for the X̄₁ logical operator
3. **LPU Z̄₁ circuit** — full gauging-measurement circuit for the Z̄₁ logical operator

**Decoder:** RelayBP (Rust-native) for all three.

### Circuit structure reminder
The LPU circuits have the form:
```
[d_init bare rounds] → [C LPU rounds] → [edge Z measurements] → [d_init bare rounds] → [data measurement]
```
The logical error rate here is the probability that the LPU protocol returns the wrong eigenvalue of X̄₁ (or Z̄₁).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

from bb_code_sim import BBCodeSimulator, BB_144_12_12, RelayBPDecoder
from surface_code_sim import ErrorModel
import gross_code_lpu_tdg as tdg

SEED = 42
print('Imports OK')
print(f'Gross code: [[{BB_144_12_12.l*BB_144_12_12.m*2}, 12, {BB_144_12_12.distance}]]')

## 1. Memory experiment — LER vs p

Standard Z-memory experiment: prepare $|\bar{0}\rangle$, run `rounds = distance = 12` syndrome rounds, measure. RelayBP decodes.

In [ ]:
# {p: shots} — BB_144_12_12 with rounds=12 is ~10s/shot, keep counts small
p_shots_mem = {
    0.003: 50,
    0.005: 40,
    0.007: 30,
    0.010: 20,
    0.015: 15,
    0.020: 10,
}

sim = BBCodeSimulator(BB_144_12_12)
mem_results = []
t0 = time.perf_counter()

for p, shots in tqdm(p_shots_mem.items(), desc='memory sweep'):
    r = sim.run(ErrorModel.symmetric(p), rounds=BB_144_12_12.distance, shots=shots, seed=SEED)
    mem_results.append(r)
    print(f'  p={p:.3f}  shots={shots}  {r}')

print(f'\nTotal: {time.perf_counter()-t0:.1f}s')

## 2. LPU X̄₁ circuit — LER vs p

Full LPU measurement circuit for X̄₁ using half-LPU l.  
Parameters: `C=10` LPU rounds, `d_init=12` bare memory rounds before/after.

In [ ]:
C, D_INIT = 10, 12

# LPU circuits are larger — use smaller shot counts
p_shots_lpu = {
    0.003: 30,
    0.005: 25,
    0.007: 20,
    0.010: 15,
    0.015: 10,
    0.020: 10,
}

x1_results = []
t0 = time.perf_counter()

for p, shots in tqdm(p_shots_lpu.items(), desc='X̄₁ sweep'):
    r = tdg.run_lpu(ErrorModel.symmetric(p), operator='X1', C=C, d_init=D_INIT,
                    shots=shots, seed=SEED)
    x1_results.append(r)
    print(f'  p={p:.3f}  shots={shots}  {r}')

print(f'\nTotal: {time.perf_counter()-t0:.1f}s')

## 3. LPU Z̄₁ circuit — LER vs p

In [ ]:
z1_results = []
t0 = time.perf_counter()

for p, shots in tqdm(p_shots_lpu.items(), desc='Z̄₁ sweep'):
    r = tdg.run_lpu(ErrorModel.symmetric(p), operator='Z1', C=C, d_init=D_INIT,
                    shots=shots, seed=SEED)
    z1_results.append(r)
    print(f'  p={p:.3f}  shots={shots}  {r}')

print(f'\nTotal: {time.perf_counter()-t0:.1f}s')

## 4. Comparison plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for results, p_shots, label, color, marker in [
    (mem_results, p_shots_mem, 'Memory (d=12 rounds)', 'steelblue', 'o'),
    (x1_results,  p_shots_lpu, f'LPU X̄₁ (C={C}, d_init={D_INIT})', 'tomato', 's'),
    (z1_results,  p_shots_lpu, f'LPU Z̄₁ (C={C}, d_init={D_INIT})', 'seagreen', '^'),
]:
    p_vals = list(p_shots.keys())
    lers   = [r.logical_error_rate    for r in results]
    errs   = [r.logical_error_rate_se for r in results]
    ax.errorbar(p_vals, lers, yerr=errs, fmt=f'{marker}-', capsize=4,
                label=label, color=color)

ax.axhline(0.5, color='grey', ls='--', lw=0.8, label='50% (random)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate')
ax.set_title('[[144,12,12]] Gross Code — LER vs $p$  (RelayBP decoder)')
ax.legend()
plt.tight_layout()
plt.show()